# GeoPlant - Single Predictor Run (imports framework)

This notebook is the example entry point for the XGBoost baseline. It keeps the workflow readable while delegating reusable logic to `geoplant_xgb/`.

On macOS, XGBoost may require `libomp` to be installed before training will run end to end.

In [ ]:
# If needed:
# !pip install -r requirements.txt
# macOS: brew install libomp
import sys

sys.path.append("./")

import numpy as np
import pandas as pd

from geoplant_xgb.config import ExperimentConfig, PredictorPairSpec
from geoplant_xgb.data import (
    align_features_with_labels,
    build_wide_labels_from_long_metadata,
    select_top_species,
    split_features_by_group,
)
from geoplant_xgb.io_csv import (
    build_features_from_meta_and_predictors_pair,
    load_metadata_csv,
    load_predictor_pairs,
)
from geoplant_xgb.metrics import macro_auc, sample_f1_at_k, sample_recall_at_k
from geoplant_xgb.model import (
    estimate_topk,
    predict_scores,
    train_ovr,
    train_richness_estimator,
)
from geoplant_xgb.predict import export_predictions


In [ ]:
# Train/Test metadata (CSV). Train contains speciesId per surveyId; test usually doesn't.
TRAIN_METADATA_CSV = "../../metadata/PA_metadata_train.csv"
TEST_METADATA_CSV = "../../metadata/PA_metadata_test.csv"

# Single predictor family (bioclimatic) — separate TRAIN and TEST files
BIOCLIM_TRAIN_CSV = "./EnvironmentalValues/Climate/PA-train-bioclimatic.csv"
BIOCLIM_TEST_CSV = "./EnvironmentalValues/Climate/PA-test-bioclimatic.csv"

RESULTS_CSV_PATH = "xgb_single_bioclim_results.csv"
SUBMISSION_CSV_PATH = "submission_predictions.csv"

In [ ]:
cfg = ExperimentConfig(
    predictor_pairs=[
        PredictorPairSpec(
            name="bioclim",
            group="climatic",
            prefix="clim_",
            train_path=BIOCLIM_TRAIN_CSV,
            test_path=BIOCLIM_TEST_CSV,
        )
    ],
    sample_id_col="survey_id",
    species_prefix="sp_",
    top_species_n=500,
    min_pos_per_species=5,
    xgb_params=dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        n_jobs=8,
        random_state=42,
        verbosity=0,
    ),
    early_stopping_rounds=30,
    ablations=[["climatic"]],
    use_richness_estimator=True,
    richness_offset=5,
)
train_meta = load_metadata_csv(TRAIN_METADATA_CSV, sample_id_col="surveyId")
test_meta = load_metadata_csv(TEST_METADATA_CSV, sample_id_col="surveyId")
train_pred, test_pred = load_predictor_pairs(cfg.predictor_pairs)
train_features, test_features = build_features_from_meta_and_predictors_pair(
    cfg, train_meta, test_meta, train_pred, test_pred
)

In [ ]:
train_labels_wide = build_wide_labels_from_long_metadata(
    train_meta,
    survey_id_column="surveyId",
    species_id_column="speciesId",
    species_prefix=cfg.species_prefix,
)
train_feat_aligned, train_lab_aligned, species_cols_all = align_features_with_labels(
    train_features, train_labels_wide, survey_id_column=cfg.sample_id_col
)
# Build placeholder labels for test with the same species columns in one concat
test_labels_placeholder = pd.concat(
    [
        test_features[[cfg.sample_id_col]].copy(),
        pd.DataFrame(0, index=test_features.index, columns=species_cols_all),
    ],
    axis=1,
)
test_feat_aligned, test_lab_aligned, _ = align_features_with_labels(
    test_features, test_labels_placeholder, survey_id_column=cfg.sample_id_col
)


In [ ]:
# Select top species and train
top_species = select_top_species(train_lab_aligned, species_cols_all, cfg)
groups = split_features_by_group(train_feat_aligned, cfg)
clim_cols = groups["climatic"]
models = train_ovr(
    train_feat_aligned[clim_cols],
    train_lab_aligned.set_index(cfg.sample_id_col),
    top_species,
    cfg,
)

# Estimate Top-K
if cfg.use_richness_estimator:
    clf, edges, bin_to_avg = train_richness_estimator(
        train_feat_aligned[clim_cols],
        train_lab_aligned.set_index(cfg.sample_id_col),
        cfg,
        nbins=15,
    )
    topk_arr = estimate_topk(
        clf, edges, bin_to_avg, test_feat_aligned[clim_cols], offset=cfg.richness_offset
    )
else:
    topk_arr = cfg.fixed_top_k

# Export predictions through the package helper
submission = export_predictions(
    models_by_species=models,
    features_matrix=test_feat_aligned[[cfg.sample_id_col] + clim_cols],
    species_column_names=top_species,
    topk_per_sample=topk_arr,
    sample_id_col=cfg.sample_id_col,
)
submission.to_csv(SUBMISSION_CSV_PATH, index=False)
submission.head()


In [ ]:
# Train-set sanity metrics (not a valid test estimate)
scores_train = predict_scores(models, train_feat_aligned[clim_cols], top_species)

fs1 = sample_f1_at_k(
    train_lab_aligned[top_species].values.astype(int), scores_train, k=25
)
rec = sample_recall_at_k(
    train_lab_aligned[top_species].values.astype(int), scores_train, k=25
)
auc = macro_auc(train_lab_aligned[top_species].values.astype(int), scores_train)

results = pd.DataFrame(
    {
        "groups": ["climatic"],
        "n_features": [len(clim_cols)],
        "n_species": [len(top_species)],
        "AUC": [auc],
        "Recall": [rec],
        "Fs1": [fs1],
        "TopK": ["per-sample"],
    }
)
results.to_csv(RESULTS_CSV_PATH, index=False)
results
